# 1. Getting started: services, tools and models

dnalm-hub serves many DNA language models from one place. Each **model family** runs as its
own service that speaks **MCP** (Model Context Protocol: JSON-RPC over HTTP). All services
offer the same core tools with the same arguments and response shapes, so switching models
means changing only the service and the `checkpoint`. A **gateway** additionally exposes an
OpenAI-compatible embeddings API (notebook 06).

| Service | Port | Models | Type |
|---|---|---|---|
| `ntv3` | 8000 | Nucleotide Transformer v3 | masked, 1 token per base |
| `ntv2` | 8001 | Nucleotide Transformer v2, v1 2.5B | masked, 6-mer tokens |
| `dnabert2` | 8002 | DNABERT-2 117M | masked, BPE tokens |
| `hyenadna` | 8003 | HyenaDNA | causal, 1 token per base |
| `evo2` | 8004 | Evo 2 7B | causal, 1 token per base |
| `generator` | 8005 | GENERator | causal, 6-mer tokens |
| `genalm` | 8006 | GENA-LM (BERT, BigBird) | masked, BPE tokens |
| `grover` | 8007 | GROVER | masked, BPE tokens |
| `gateway` | 8080 | OpenAI-compatible `/v1/embeddings` over all of the above | |

**The series**

1. Getting started (this notebook): connect, discover tools and checkpoints, first embedding
2. [Embeddings](02_embeddings.ipynb): batches, per-token output, layers, a small classifier
3. [Variant effects](03_variant_effects.ipynb): zero-shot SNP scores, masked prediction, a mutation scan
4. [Generation](04_generation.ipynb): continuing DNA with the causal models
5. [NTv3 species models](05_ntv3_annotation.ipynb): genome annotation and experimental tracks
6. [OpenAI-compatible gateway](06_openai_gateway.ipynb): embeddings with any OpenAI client

The notebooks are only clients: the models run on the server.

## Setup

Two environment variables tell the notebooks where the server is:

- `MCP_HOST`: e.g. `http://my-gpu-server` (default `http://localhost`),
- `MCP_AUTH_TOKEN`: the shared token from the server's `.env`; without it a protected server
  answers `401 Unauthorized`.

`tutorial_utils.py` (next to this notebook) creates one `GenericMcpClient` per service. The
client is in this repository (`packages/dnalm-client`) and uses only the Python standard
library. Start Jupyter from the repo root with:

```bash
uv run --project packages/dnalm-client --with jupyterlab --with matplotlib --with openai \
    jupyter lab examples/tutorial
```

In [1]:
from tutorial_utils import HBB, connect, show_table

clients = connect()

server http://localhost | token set: True | up: ntv3, ntv2, dnabert2, hyenadna, evo2, generator, genalm, grover


## Which tools does each service offer?

`list_tools()` is the standard MCP `tools/list` request. The core tools exist everywhere;
a few are specific to a model family.

In [2]:
tools = {name: {t["name"] for t in c.list_tools()} for name, c in clients.items()}
core = set.intersection(*tools.values())
print("on every service:", ", ".join(sorted(core)))
for tool in sorted(set().union(*tools.values()) - core):
    print(f"{tool:<26} only on: {', '.join(s for s, names in tools.items() if tool in names)}")

on every service: compare_sequences, embed_sequence, get_embedding_layers, get_model_info, list_available_checkpoints, score_snp
annotate_sequence          only on: ntv3
generate_sequence          only on: hyenadna, evo2, generator
list_species               only on: ntv3
list_tracks                only on: ntv3
predict_masked_positions   only on: ntv3
predict_tracks             only on: ntv3


- `predict_masked_positions` needs one token per base and a masked model, so only NTv3 has it.
- `generate_sequence` exists on the causal (autoregressive) models.
- `annotate_sequence`, `predict_tracks`, `list_species`, `list_tracks` belong to NTv3's
  species-conditioned `post` checkpoints (notebook 05).

Every tool has a docstring, which MCP clients (and LLM agents) see as its description:

In [3]:
for t in clients["ntv2"].list_tools():
    if t["name"] == "embed_sequence":
        print(t["description"])

Return NTv2 embeddings for one or more DNA sequences.

    Args:
        sequence: A single DNA sequence, or a list of sequences to embed as a batch
            (letters A/C/G/T/N only). Each must fit the checkpoint's token limit.
        checkpoint: A name from list_available_checkpoints, or a full HuggingFace repo id.
        layer_name: Hidden-state layer to read, as an integer string (e.g. "6") or
            "last". See get_embedding_layers for valid values.
        pooling: "mean" (default) averages over all non-padding tokens into one vector
            per sequence, as in InstaDeep's model card. "per_token" returns one row per
            TOKEN -- a leading <cls> then one row per 6-mer (or leftover single base) --
            not one row per nucleotide as in ntv3-mcp.
    


## Which models are available?

`list_available_checkpoints` lists the registered checkpoints of a service. Each tool takes
a `checkpoint` argument: one of these aliases, or (for most services) any compatible
HuggingFace repo id. Without `checkpoint`, the service's default is used.

In [4]:
for name, client in clients.items():
    rows = []
    for ck in client.list_available_checkpoints():
        rows.append(
            {
                "checkpoint": ck["name"] + (" *" if ck.get("default") else ""),
                "params": ck["params"],
                "kind": ck.get("stage") or ck.get("generation") or ck.get("architecture") or "",
                "context": ck.get("context") or f"{ck.get('max_tokens')} tokens",
            }
        )
    print(f"== {name}  (* = default)")
    show_table(rows)
    print()

== ntv3  (* = default)
checkpoint       params  kind  context    
---------------  ------  ----  -----------
8m-pre           7.7M    pre   full (~1Mb)
8m-pre-8kb       7.7M    pre   8kb        
100m-pre *       0.1B    pre   full (~1Mb)
100m-pre-8kb     0.1B    pre   8kb        
100m-post        0.1B    post  full (~1Mb)
100m-post-131kb  0.1B    post  131kb      
650m-pre         0.7B    pre   full (~1Mb)
650m-pre-8kb     0.7B    pre   8kb        
650m-post        0.7B    post  full (~1Mb)
650m-post-131kb  0.7B    post  131kb      
5ds-pre          0.6B    pre   full (~1Mb)
5ds-pre-8kb      0.6B    pre   8kb        
5ds-post         0.6B    post  full (~1Mb)
5ds-post-131kb   0.6B    post  131kb      

== ntv2  (* = default)
checkpoint  params  kind  context    
----------  ------  ----  -----------
50m         50M     v2    2048 tokens
100m        100M    v2    2048 tokens
250m        250M    v2    2048 tokens
500m *      500M    v2    2048 tokens
2.5b-v1     2.5B    v1    1000 tokens

## Model details

`get_model_info` loads a checkpoint (the first call downloads it, which can take minutes for
the large ones) and reports its architecture.

In [5]:
info = clients["ntv2"].get_model_info(checkpoint="50m")
for key in [
    "repo_id",
    "device",
    "dtype",
    "num_parameters",
    "hidden_size",
    "num_hidden_state_layers",
    "max_tokens",
]:
    print(f"{key:<24} {info.get(key)}")

repo_id                  InstaDeepAI/nucleotide-transformer-v2-50m-multi-species
device                   cuda
dtype                    float32
num_parameters           55904972
hidden_size              512
num_hidden_state_layers  13
max_tokens               2048


## A first embedding

`embed_sequence` turns a DNA sequence into a vector. `HBB` is the first 90 bp of the human
beta-globin coding sequence (defined in `tutorial_utils.py`).

In [6]:
out = clients["ntv2"].embed_sequence(HBB, checkpoint="50m")
print("checkpoint:", out["checkpoint"])
print("layer:", out["layer_name"], "of", out["num_layers"], "| pooling:", out["pooling"])
print(
    "dimension:",
    len(out["embedding"]),
    "| first values:",
    [round(x, 3) for x in out["embedding"][:4]],
)

checkpoint: InstaDeepAI/nucleotide-transformer-v2-50m-multi-species
layer: 12 of 12 | pooling: mean
dimension: 512 | first values: [-0.106, 0.12, 0.004, 0.1]


## Errors are readable

Bad input raises `RuntimeError` with the server's message: invalid characters, an unknown
checkpoint, or a sequence over the model's context limit. Sequences are never truncated
silently.

In [7]:
for bad in ["ACGTXACGT", "ACGTAC" * 2100]:
    try:
        clients["ntv2"].embed_sequence(bad, checkpoint="50m")
    except RuntimeError as exc:
        print(exc, "\n")

Error executing tool embed_sequence: Sequence contains invalid character(s) ['X']; only ['A', 'C', 'G', 'N', 'T'] are allowed. 

Error executing tool embed_sequence: Sequence 0 tokenizes to 2101 tokens, above this checkpoint's 2048-token context limit (roughly 12282 bp of N-free sequence; each 'N' costs a whole token). Sequences are never truncated -- split it into shorter windows. 



## Without Python: plain HTTP

MCP is JSON-RPC over HTTP, so `curl` works too. Open a session, then call tools with the
`Mcp-Session-Id` header; answers arrive as Server-Sent Events (`data: {...}`):

```bash
URL="$MCP_HOST:8001/mcp"
HDR=(-H "Authorization: Bearer $MCP_AUTH_TOKEN" -H "Content-Type: application/json"
     -H "Accept: application/json, text/event-stream")

SID=$(curl -si "$URL" "${HDR[@]}" -d '{"jsonrpc":"2.0","id":1,"method":"initialize",
  "params":{"protocolVersion":"2025-06-18","capabilities":{},"clientInfo":{"name":"curl","version":"1"}}}' \
  | grep -i '^mcp-session-id:' | cut -d' ' -f2 | tr -d '\r')
curl -s "$URL" "${HDR[@]}" -H "Mcp-Session-Id: $SID" -d '{"jsonrpc":"2.0","method":"notifications/initialized"}'

curl -s "$URL" "${HDR[@]}" -H "Mcp-Session-Id: $SID" -d '{"jsonrpc":"2.0","id":2,"method":"tools/call",
  "params":{"name":"embed_sequence","arguments":{"sequence":"ACGTACGTACGT","checkpoint":"50m"}}}'
```

To use the services from an MCP client such as Claude Code, see
[`examples/mcp_client_config.json`](../mcp_client_config.json).

**Next:** [2. Embeddings](02_embeddings.ipynb)